In [0]:
## CREATING Dim_cust table

from pyspark.sql.functions import *
from pyspark.sql.types import *

df = spark.read.format("csv").option("header",True).load("/Volumes/transportation/bronze/source_data/customers_day1.csv")

display(df)


In [0]:
#customer_sk INT,
#effective_start_date DATE,
#effective_end_date DATE,
#is_current BOOLEAN


add_new_columns = df.withColumn("effective_start_date",current_date())\
    .withColumn("effective_end_date",to_date(lit("9999-12-31")))\
    .withColumn("is_current",lit(True))


#cust_sk
sk = spark.read.table("customer_dim").agg(max("customer_sk")).collect()[0][0]
sk_new = 1 if sk is None else sk+1
add_new_columns = add_new_columns.withColumn("customer_sk",(monotonically_increasing_id()+sk_new).cast("int"))

final_df = add_new_columns.select("customer_sk","customer_id","customer_name","city","status","effective_start_date","effective_end_date","is_current")

final_df.write.mode("append").saveAsTable("customer_dim")




In [0]:
%sql

create table if not exists customer_dim(
customer_sk INT,
customer_id STRING,
customer_name STRING,
city STRING,
status STRING,
effective_start_date DATE,
effective_end_date DATE,
is_current STRING
)  ;


In [0]:
%sql
select * from customer_dim

In [0]:
## SCD TYPE 2

data_day2 = [
 (1001, "Anita Sharma", "Mumbai", "Active"), # unchanged
 (1002, "Ravi Kumar", "Hyderabad", "Active"), # city changed
 (1003, "Priya Singh", "Bengaluru", "Inactive"), # status changed
 (1004, "Sanjay Rao", "Pune", "Active"), # new
]
cols = ["customer_id","customer_name","city","status"]

df_day2 = spark.createDataFrame(data_day2,cols)
df_day2 = df_day2.withColumn("start_date",current_date())
df_day2.createOrReplaceTempView("day2_temp")

In [0]:
%sql

select * from  customer_dim

In [0]:
%sql
select * from day2_temp

In [0]:
%sql
merge into customer_dim a
using day2_temp b
on a.customer_id = b.customer_id
and a.is_current = "true"
when matched and (a.customer_name <> b.customer_name or a.city <> b.city or a.status <> b.status)
then update set 
a.effective_end_date = b.start_date,
a.is_current = "false"
when not matched then 
insert (customer_sk,customer_id,customer_name,city,status,effective_start_date,effective_end_date,is_current)
values (
(select max(customer_sk)+1 from customer_dim),
b.customer_id,
b.customer_name,
b.city,
b.status,
b.start_date,
"9999-12-31",
"True"
)


In [0]:
%sql
select customer_id from customer_dim where is_current = 'true' group by customer_id having count(1) > 1

In [0]:
%sql
SELECT customer_id, count(*) AS current_count
FROM dim_customer
WHERE is_current = true
GROUP BY customer_id
HAVING current_count <> 1;